# Is the prefix signal additive over letters?

Registered test (B1-B3 committed before the run; see README). Same split as the sibling experiments: 2185 stationary census rules train, 162 fresh stationary rules test, disjoint. Ridge regression (penalty fixed at 1.0, not tuned) on binary 'letter is L' indicators for positions 2-8, with and without all pairwise products, against the 8-letter lookup P8.

**Assumptions.** Same stationarity filter and fresh-rule target as the W3c audit; binary indicators only.

In [1]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
d = json.loads((root / "experiments/archive/2026-10-kappa-prefix-additive/results/additive.json").read_text())
print("inputs:", {k: v[:12] for k, v in d["input_sha256"].items()}, "train", d["n_train"], "test", d["n_test"], "overlap", d["overlap"])
for k, v in d["r2"].items():
    lo, hi = d["ci"][k]
    print(f"{k:3s} R^2 {v:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
print(d["verdicts"])
assert d["overlap"] == 0
assert list(d["verdicts"].values()) == [True, False, True]  # B1 held, B2 failed, B3 held, as in the README
assert d["r2"]["A"] < 0.2 and 0.3 < d["r2"]["A2"] < 0.4 and abs(d["r2"]["P8"] - 0.624) < 2e-3

inputs: {'probe_summary.csv': 'c76c7927185b', 'prefix_audit_predictions.csv': '742f0fe1d776'} train 2185 test 162 overlap 0
A   R^2 +0.109  95% CI [+0.003, +0.194]
A2  R^2 +0.351  95% CI [+0.234, +0.447]
P8  R^2 +0.624  95% CI [+0.473, +0.729]
{'B1 additive A < 0.40': True, 'B2 pairwise A2 >= 0.50': False, 'B3 P8 - A >= 0.15': True}


## What this shows and does not show
- An additive model over letter positions scores 0.11; adding all pairwise interactions gives 0.35 (B2, which predicted at least 0.50, failed); the lookup scores 0.62. The signal needs interactions of order three or higher among letters 2-8.
- **Not shown:** whether that is real interaction structure or memorisation of prefix families that overlap between training and test. The ridge penalty was not tuned and only binary indicators were used.

## Next
A split by prefix family (hold out whole 6-letter prefix families) to see whether any of the 0.62 survives without family overlap.